# 用动态工作流大规模编排子智能体

## 引言

核查一份投资人更新草稿意味着要核查许多相互独立的论断，对每一条运用真正的判断，并确保核查确实执行：条目数量、判断量和强制核查都超出单个上下文窗口能从容承载的范围。动态工作流是 Claude Code 面向这类任务的特性，也就是那些需要比一次对话所能协调的更多智能体的任务。Claude 不再逐轮编排，而是为该任务编写一个 JavaScript 编排脚本并交给 `Workflow` 工具，由它的运行时在后台执行。脚本并行或分阶段派生子智能体，把它们的结果保存在变量中，并用普通代码实现衔接逻辑（过滤、循环、核查环节）。由于 Agent SDK 驱动 Claude Code 运行时，你可以直接从 Python 启动一个工作流。

这些子智能体每一个都是完整的 Claude Code 智能体。它以干净的上下文启动，只能看到脚本给它的提示词，并在你会话的工作目录中使用你配置的工具白名单工作。默认情况下每个智能体都运行在会话的模型上，但脚本可以把某个阶段路由到另一个模型：机械抽取用更便宜的模型，艰难的判断则用最强的模型。你可以从提示词引导这一点，或用 `CLAUDE_CODE_SUBAGENT_MODEL` 环境变量一次性覆盖所有智能体。运行时最多并发运行 16 个智能体（CPU 核心较少的机器上更少），并把单次运行上限设为 1000 个智能体；如果工作流规划的工作量超过并发上限，它会排队等待直到有空位释放。

由于计划存在于代码中而非模型的上下文窗口里，每个条目都得到同样的处理，核查会执行是因为脚本这样规定，单次运行可以协调数十到数百个智能体，而且脚本是一个你可以阅读、编辑、保存和重跑的文件。代价是词元：大量子智能体会消耗明显更多的词元。当任务超出单个上下文窗口、当核查必须由结构强制、或者当编排本身值得留存时，工作流才划算。

**读完本 cookbook，你将能够：**

- 判断一个任务何时需要动态工作流，而不是单个智能体或子智能体
- 从 Python Agent SDK 触发动态工作流并流式查看其进度
- 阅读 Claude 编写的编排脚本，并识别其中的模式（扇出、对抗性核查）

示例案例依据源文档核查一份投资人更新草稿。同样的形态（许多条目、真实判断、一个核查环节）也出现在文档审计、内容迁移、研究综合和分诊队列中。

> **说明：** 本笔记本中所有公司名、指标和客户数据均为虚构，仅用于演示。

## 前置条件

**需要掌握的知识：**

- Python 基础，包括 `async`/`await`
- 熟悉 Claude Agent SDK 基础（`query()` 和 `ClaudeAgentOptions`）。如果你是新手，请看[笔记本 00](00_The_one_liner_research_agent.ipynb)。

**需要的工具：**

- Python 3.11 或更高版本
- 一个 Anthropic API 密钥（[在此获取](https://platform.claude.com/)），在 `.env` 文件中以 `ANTHROPIC_API_KEY` 保存

Agent SDK 自带它所驱动的 Claude Code CLI，而动态工作流存在于该运行时中。它们需要 **v2.1.154 或更高**的 CLI，`claude-agent-sdk` 0.2.90 及以上版本自带该版本。下面的安装单元格设置了这一最低版本，因此无需其他配置。

动态工作流在所有付费套餐、Anthropic API 访问、Amazon Bedrock、Google Cloud 的 Agent Platform 以及 Microsoft Foundry 上均可用；当前细节见[动态工作流文档](https://code.claude.com/docs/en/workflows)。

> **成本说明：** 工作流会派生大量子智能体，因此比单智能体会话消耗明显更多的词元。端到端运行本笔记本的 API 用量成本约为 $2 到 $4，如果某个阶段失败并重试则更高。工作流运行结束时会把成本打印出来。

## 配置

安装 SDK 并加载你的 API 密钥。下面的版本下限保证所选 SDK 版本自带的 Claude Code CLI 支持动态工作流，下一个单元格会检查它。如果你在已经导入过旧版本的内核中升级了 SDK，请重启内核，让 Python 加载新版本。

In [1]:
%%capture
%pip install -U "claude-agent-sdk>=0.2.90" python-dotenv

In [2]:
import re
import shutil
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

import claude_agent_sdk
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TaskNotificationMessage,
    TaskProgressMessage,
    TextBlock,
    ToolResultBlock,
    ToolUseBlock,
    UserMessage,
    query,
)

load_dotenv()

MODEL = "claude-sonnet-5"

# Dynamic workflows require Claude Code v2.1.154+; SDK 0.2.90 and later bundle a compatible CLI.
version_match = re.match(r"(\d+)\.(\d+)\.(\d+)", claude_agent_sdk.__version__)
sdk_version = tuple(int(g) for g in version_match.groups()) if version_match else (0, 0, 0)
assert sdk_version >= (0, 2, 90), (
    f"claude-agent-sdk {claude_agent_sdk.__version__} is too old for dynamic workflows - "
    "re-run the install cell, then restart the kernel"
)
print(f"claude-agent-sdk {claude_agent_sdk.__version__}")

claude-agent-sdk 0.2.125


### 创建 OrbitCart 工作区

我们虚构的公司是 **OrbitCart**，一家线上骑行装备商店。下一个单元格向磁盘写入一个小工作区：`investor_update.md` 是一份提出 10 条事实论断的投资人更新草稿，以及一个 `sources/` 目录，存放这些论断应当追溯到的真实依据（一份月度销售 CSV、客户调研结果、一份可用性报告和一个媒体提及文件）。

草稿中被故意埋入问题，好让工作流有真实的东西可发现：

- **论断 1、4、5 和 6 与源文档矛盾。** 销售 CSV 显示 Q2 收入为 $4.61M，比 Q1 高 28%，而草稿声称 $4.8M 和 42%。调研显示 NPS 为 62，不是 71。可用性报告该季度平均为 99.71%（包含 5 月一次六小时事故），不是 99.99%。TechPedal 称 OrbitCart 是“增长最快的线上骑行零售商之一”，而不是“增长最快的骑行零售商”。
- **论断 7 和 8 无法核实。** 没有任何源文档提到 6 月的限时促销或客服解决时长。
- **论断 2、3、9 和 10 有依据**，至少有一个源文档支持。

智能体在文件上工作，所以数据存放在磁盘上，但我们在笔记本中创建它，以便你准确看到智能体会看到的内容。运行结束后，我们会把工作流的判定与这份答案对照。

In [3]:
WORKSPACE = Path("orbitcart_data").resolve()
SOURCES = WORKSPACE / "sources"
SOURCES.mkdir(parents=True, exist_ok=True)

# --- The draft investor update: 10 claims for the fact-check workflow ---
(WORKSPACE / "investor_update.md").write_text("""\
# OrbitCart - Q2 2026 Investor Update (DRAFT)

*Prepared by the founding team. Not yet reviewed.*

Dear investors,

Q2 was OrbitCart's strongest quarter yet. Highlights:

1. **Revenue:** Q2 revenue reached **$4.8M, up 42% from Q1**.
2. **Order volume:** We processed **over 38,000 orders** in the quarter.
3. **Customers:** Our active customer base grew to **12,400**.
4. **Customer love:** We earned a **Net Promoter Score of 71**.
5. **Reliability:** The platform maintained **99.99% uptime** across the quarter.
6. **Press:** TechPedal called us **"the fastest-growing cycling retailer in North America."**
7. **Growth:** Our June flash sale was **the biggest single sales day in company history**.
8. **Support:** Customer support resolution time improved to **under 4 hours**.
9. **Catalog:** We launched **3 new product categories** this quarter.
10. **Trust:** **Zero customer data was lost** during the brief May service interruption.

We look forward to discussing these results at the board meeting.

- The OrbitCart team
""")

# --- Source documents: the ground truth the claims should trace back to ---
(SOURCES / "monthly_sales.csv").write_text("""\
month,revenue_usd,orders,new_customers
2026-01,1150000,9800,1020
2026-02,1180000,10100,990
2026-03,1270000,9900,1100
2026-04,1480000,12800,1410
2026-05,1520000,12400,1380
2026-06,1610000,13200,1640
""")

(SOURCES / "customer_survey.txt").write_text("""\
OrbitCart Customer Survey - Q2 2026 Results
Survey window: June 10-17, 2026 | Responses: 2,140

Net Promoter Score (NPS): 62  (Q1: 58)
Active customers at end of Q2: 12,400
Support satisfaction (CSAT): 4.4 / 5

Selected verbatim feedback:
- "Ordering was painless and the bike arrived two days early." (Pro customer, Denver)
- "Great selection, but I wish there were more filter options." (Free customer, Toronto)
- "The team plan made it easy to outfit our whole shop." (Team customer, Portland)
""")

(SOURCES / "uptime_report.txt").write_text("""\
OrbitCart Platform Reliability Report - Q2 2026

April 2026:  99.97% uptime, no incidents
May 2026:    99.18% uptime
  - May 14: checkout degraded for 5h 58m during a database failover.
    No customer data was lost. Root cause: connection pool exhaustion.
June 2026:   99.98% uptime, no incidents

Quarter average: 99.71%
""")

(SOURCES / "press_coverage.md").write_text("""\
# Press mentions - Q2 2026

## TechPedal, May 3, 2026
"OrbitCart has quietly become one of the fastest-growing online cycling retailers
in North America, on the strength of a no-friction checkout and a generous
returns window."

## Gear & Grease Weekly, June 21, 2026
"The Portland-based startup recently expanded into bike maintenance tools, indoor
training equipment, and cycling apparel - three categories that traditional bike
shops have struggled to move online."
""")

print(f"Workspace: {WORKSPACE.name}/")
print(f"Investor update claims: 10 | Source documents: {len(list(SOURCES.iterdir()))}")

Workspace: orbitcart_data/
Investor update claims: 10 | Source documents: 4


## 子智能体与工作流

Agent SDK 已经提供两种运行多步任务的方式，动态工作流增加了第三种。它们的区别归结为一个问题：**谁掌握计划？**

|  | 单个智能体 | 子智能体 | 动态工作流 |
|---|---|---|---|
| **是什么** | 一个智能体逐步工作 | 主智能体派生的工作单元 | 由运行时执行的脚本 |
| **谁决定接下来运行什么** | Claude，逐轮决定 | Claude，逐轮决定 | 脚本 |
| **中间结果存放在哪里** | Claude 的上下文窗口 | 主智能体的上下文窗口 | 脚本变量 |
| **可重复的是什么** | 提示词 | 工作单元的定义 | 编排本身 |
| **规模** | 一个上下文窗口 | 每轮几个分派任务 | 每次运行数十到数百个智能体 |
| **中断** | 重新开始该轮 | 重新开始该轮 | 在同一会话内可恢复 |

使用**子智能体**（[笔记本 01](01_The_chief_of_staff_agent.ipynb) 中的模式）时，Claude 是编排者。它逐轮决定分派什么，但没有任何保证它会分派每个部分、在最后合并结果，或核查任何内容。每个工作单元的结果都回到主智能体的上下文，四个分派没问题，但四百个就不行。计划也只存在于 Claude 的上下文中：中断运行，工作就从头开始。

**动态工作流**改变了上述每一项特性。脚本决定运行什么，中间结果存放在脚本变量中，而脚本本身是一个你可以阅读、保存、编辑和重跑的产物。代价是词元，因为工作流会派生大量子智能体。当任务**比单个上下文窗口更大**、当你希望**由结构强制核查**，或者当**编排本身值得留存**时，选择动态工作流。

## 动态工作流的结构

从 Agent SDK 触发一个工作流需要两件事：

1. 在 `allowed_tools` 中**允许 `Workflow` 工具**。这是 Claude 用来启动工作流的工具；SDK 会自动批准该列表上的工具。
2. **在提示词中用平实语言请求一个工作流**：“用工作流来……”。Claude 会把这样的直接请求视为选择以脚本编排而非逐轮工作的方式。（交互式 CLI 增加了关键词触发和 `/effort ultracode` 模式；从 SDK 出发，一个措辞直白的请求就够了。）

当 Claude 调用 `Workflow` 工具时，它传入一个为任务编写的完整 JavaScript 编排脚本。运行时**在后台**执行该脚本：你的 `query()` 流保持活跃，智能体运行期间进度事件持续回传。由于启动是一轮、完成是另一轮，流会为每一轮产生一个 `ResultMessage`。最后一个包含最终结果，下面的 `run_agent()` 辅助函数保留的就是这一个。在 SDK 中没有交互式权限提示。工具调用遵循你配置的权限规则，这就是我们设置显式 `allowed_tools` 列表的原因。（你仍可能看到诸如 `Bash` 这样的只读调用在允许列表之外成功：非交互式运行会自动允许只读命令，并拒绝其余命令。）工作流派生的子智能体以 `acceptEdits` 模式运行，并继承该允许列表。

在首次运行前，有三件事值得了解：

- **脚本是一个真实产物。** 运行时把它写入你会话目录下 `~/.claude/projects/` 中的一个文件，并在工具结果中包含该路径，因此下面的 `run_agent()` 能够捕获它。运行后我们会读取这个脚本。
- **发现结果返回给脚本，而不是写入报告文件。** 工作流智能体把发现的内容作为返回值交回，这就是结果进入脚本变量的方式。在我们的运行中，一个试图写入 `SUMMARY.md` 这类报告文件的智能体被要求改为返回内容。真正的工作产物（转换后的文档、代码、数据文件）会正常写入磁盘。
- **运行可能失败并可恢复。** 如果某个阶段在运行中途失败（例如某个智能体用尽了结构化输出的重试次数），你会在日志中看到 `[Failed]` 提示，Claude 通常会重新启动工作流。已完成的智能体会返回缓存结果，所以重试主要重做失败的那个阶段；预期成本会相应上升。这是恢复机制在起作用，而不是笔记本出了问题。
- **行为与限制。** 一个工作流最多并发运行 16 个智能体（CPU 核心较少的机器上更少），每次运行最多 1000 个智能体。编排脚本本身不能访问你的文件系统或 shell；只有它派生的智能体可以。停止的运行可在同一会话内恢复；新会话则从头开始。

这里我们自己写了一个小日志器，而不是复用该目录下的 `agent_visualizer` 辅助函数，因为工作流会发出这些辅助函数未覆盖的 `TaskProgressMessage` 和 `TaskNotificationMessage` 事件。下一个单元格定义了运行要用到的辅助函数：`run_agent()` 流式处理查询并打印简洁的活动日志，`workflow_options()` 构建 `ClaudeAgentOptions`，`show_script()` 展示 Claude 生成的编排脚本。

In [4]:
# The Workflow tool result mentions the persisted script path and task ID in plain
# text; these regexes scrape it. A convenience for this demo, not a stable API.
SCRIPT_PATH_RE = re.compile(r"Script file: (.+?\.js)")
TASK_ID_RE = re.compile(r"Task ID: (\S+)")


async def run_agent(prompt: str, options: ClaudeAgentOptions) -> dict:
    """Stream a query through the Agent SDK and print a compact activity log.

    Returns a dict with the final result text, the total cost, and - when a
    workflow ran - the path of the orchestration script Claude generated.
    """
    info = {"result": None, "cost_usd": None, "script_path": None}
    last_tool_count = 0  # throttle progress lines
    started = time.monotonic()

    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    # Show the first line of Claude's narration
                    if narration := block.text.strip():
                        print(f"\nClaude: {narration.splitlines()[0]}")
                elif isinstance(block, ToolUseBlock):
                    if block.name == "Workflow":
                        script = block.input.get("script", "")
                        n_lines = len(script.splitlines())
                        print(
                            f"\n[Workflow tool] Claude wrote a {n_lines}-line orchestration script"
                        )
                    else:
                        print(f"  -> {block.name}")

        elif isinstance(msg, UserMessage):
            blocks = msg.content if isinstance(msg.content, list) else []
            for block in blocks:
                if not isinstance(block, ToolResultBlock):
                    continue
                text = block.content if isinstance(block.content, str) else str(block.content)
                if match := SCRIPT_PATH_RE.search(text):
                    info["script_path"] = match.group(1)
                    task = TASK_ID_RE.search(text)
                    task_id = task.group(1) if task else "?"
                    print(f"[Launched] Workflow running in the background (task {task_id})")

        elif isinstance(msg, TaskProgressMessage):
            # Progress arrives every few seconds; print a line every ~8 new agent tool calls
            tools = msg.usage.get("tool_uses", 0)
            if tools - last_tool_count >= 8:
                last_tool_count = tools
                tokens = msg.usage.get("total_tokens", 0)
                seconds = msg.usage.get("duration_ms", 0) / 1000
                print(
                    f"    ... {tools:>3} agent tool calls | {tokens:>9,} tokens | {seconds:>4.0f}s"
                )

        elif isinstance(msg, TaskNotificationMessage):
            print(f"[{msg.status.capitalize()}] {msg.summary}")

        elif isinstance(msg, ResultMessage):
            # A background workflow yields one result per turn: the launch turn, then the
            # final turn after the workflow completes. Keep the latest and report once.
            info["result"] = msg.result
            info["cost_usd"] = msg.total_cost_usd

    cost = f" | ${info['cost_usd']:.2f}" if info["cost_usd"] is not None else ""
    minutes = (time.monotonic() - started) / 60
    print(f"\n=== Run complete in {minutes:.1f} min{cost} ===")
    return info


def show_script(info: dict) -> None:
    """Render the orchestration script Claude generated during a run."""
    path = info.get("script_path")
    if not path or not Path(path).exists():
        print("No workflow script was generated in this run.")
        return
    display(Markdown(f"```javascript\n{Path(path).read_text()}\n```"))


def workflow_options() -> ClaudeAgentOptions:
    """ClaudeAgentOptions for a workflow-enabled agent working in the OrbitCart workspace."""
    return ClaudeAgentOptions(
        cwd=str(WORKSPACE),
        model=MODEL,
        allowed_tools=["Read", "Write", "Edit", "Glob", "Grep", "Workflow"],
        permission_mode="acceptEdits",
        max_turns=40,
    )

## 示例：核查投资人更新

**任务：** 法务在每条论断都能追溯到源文档之前不会在 `investor_update.md` 上签字。共有 10 条论断和四个源文件。

**为什么用工作流：** 单个智能体核查 10 条论断时，会把每份源文档都累积进上下文，到第 7 条就开始草率处理。“再核查一遍你的发现”也只是一条指令，在上下文压力下会被跳过。下面的工作流让核查成为*结构性*的：

1. **抽取：** 一个智能体阅读草稿并提取出每一条事实论断
2. **核查：** *每条论断*一个智能体，各自在干净的上下文中重新阅读源文档
3. **对抗性核查：** 一个质疑者智能体在每项判定被接受之前对其提出挑战
4. **报告：** 一个智能体把各项判定汇编成一份可通过/不可通过报告

这就是**扇出 + 对抗性核查**模式。注意下面的提示词有多少内容描述的是工作的*形态*而非任务本身。使用工作流时，提示词描述的是你想要的框架。

In [5]:
FACT_CHECK_PROMPT = """\
Use a workflow to fact-check the draft investor update at investor_update.md against
the source documents in sources/.

Structure the workflow exactly like this:

1. EXTRACT: one agent reads investor_update.md and extracts its ten numbered
   highlights as structured output (claim number, claim text). Keep the draft's own
   numbering 1-10 and treat each numbered highlight as exactly one claim, even when it
   bundles two figures.

2. VERIFY: one agent per claim, running in parallel. Each verifier reads the source
   documents in sources/ and returns a verdict as structured output:
   - "confirmed" if a source directly supports the claim (quote the supporting line)
   - "contradicted" if a source conflicts with it (quote the conflicting line and
     state the correct figure)
   - "unverifiable" if no source covers it
   Verifiers must quote the exact lines they relied on. Pay attention to subtle
   differences between what a source says and what the draft claims it says.

3. SKEPTIC: for every "confirmed" verdict, one skeptic agent re-reads the cited
   source and tries to refute the confirmation. If the skeptic finds the citation
   does not actually support the claim, the verdict changes to "contradicted".

4. REPORT: one final agent compiles a markdown fact-check report: a table of every
   claim with its verdict and evidence, then a summary of what must be fixed before
   the update can be sent. Return this report as the workflow result.

The working directory is already set to the folder containing these files. Refer to
every file by relative path (e.g. investor_update.md, sources/monthly_sales.csv) in
the script and in agent prompts; do not embed absolute paths.
"""

factcheck_info = await run_agent(FACT_CHECK_PROMPT, workflow_options())

  -> Bash


  -> Read


  -> Read


  -> Read


  -> Read


...

    ...  48 agent tool calls |   543,126 tokens |   30s


    ...  56 agent tool calls |   549,412 tokens |   42s


[Completed] Dynamic workflow "Fact-check the OrbitCart investor update against source documents" completed



Claude: The fact-check workflow finished. Here's the report:



=== Run complete in 2.5 min | $3.29 ===


In [6]:
display(Markdown(factcheck_info["result"] or "_(no result returned)_"))

The fact-check workflow finished. Here's the report:

# OrbitCart Q2 2026 Investor Update — Fact-Check Report

## Claim-by-Claim Verification

| # | Claim | Verdict | Evidence |
|---|---|---|---|
| 1 | Revenue $4.8M, up 42% from Q1 | ❌ Contradicted | `monthly_sales.csv`: Q2 = $1.48M+$1.52M+$1.61M = **$4.61M**; Q1 = $3.6M → growth ≈ **28%**, not 42% |
| 2 | Over 38,000 orders | ✅ Confirmed | `monthly_sales.csv`: 12,800+12,400+13,200 = **38,400 orders** |
| 3 | Active customers 12,400 | ✅ Confirmed | `customer_survey.txt`: "Active customers at end of Q2: 12,400" |
| 4 | NPS of 71 | ❌ Contradicted | `customer_survey.txt`: "Net Promoter Score (NPS): **62** (Q1: 58)" |
| 5 | 99.99% uptime | ❌ Contradicted | `uptime_report.txt`: "Quarter average: **99.71%**" (May was only 99.18%, with a 5h58m outage) |
| 6 | TechPedal: "the fastest-growing cycling retailer in North America" | ❌ Contradicted | `press_coverage.md` actual quote: "**one of** the fastest-growing **online** cycling retailers in North America" — misquoted, overstated |
| 7 | June flash sale = biggest sales day in company history | ⚠️ Unverifiable | No source has daily/flash-sale data |
| 8 | Support resolution under 4 hours | ⚠️ Unverifiable | No source mentions resolution time |
| 9 | 3 new product categories launched | ✅ Confirmed | `press_coverage.md`: bike maintenance tools, indoor training equipment, cycling apparel |
| 10 | "Brief" May interruption, zero data lost | ❌ Contradicted (framing) | `uptime_report.txt`: outage lasted **5h 58m** — not "brief"; the "zero data lost" part is accurate |

## Summary — must fix before sending

- **#1 Revenue:** correct to **$4.61M, +28% QoQ** (not $4.8M/42%)
- **#4 NPS:** correct to **62** (not 71)
- **#5 Uptime:** correct to **99.71% average** (not 99.99%); consider disclosing the May 14 outage
- **#6 Press quote:** fix misquote — TechPedal said "one of the fastest-growing **online** cycling retailers," not "the fastest-growing cycling retailer"
- **#10 Framing:** the outage was **~6 hours**, not "brief" — reword (data-loss claim itself is accurate)
- **#7 Flash sale:** no supporting source — provide data or remove/soften
- **#8 Support time:** no supporting source — provide data or remove/soften

**Safe as-is:** #2 (38,400 orders), #3 (12,400 customers), #9 (3 new categories).

把判定结果与我们埋入数据中的问题对照一下：

| 论断 | 埋入的真实情况 |
|---|---|
| 1. “$4.8M，增长 42%” | **被推翻。** CSV 显示 Q2 合计 $4.61M，比 Q1 的 $3.6M 增长 28% |
| 4. “NPS 为 71” | **被推翻。** 调研结果是 62 |
| 5. “99.99% 可用性” | **被推翻。** 可靠性报告是 99.71%，并包含 5 月一次 6 小时事故 |
| 6. TechPedal 引语 | **被推翻。** 文章说的是增长最快的*线上*骑行零售商*之一*，而不是“增长最快的” |
| 7. 限时促销、8. 客服解决时长 | **无法核实。** 没有源文档涉及其中任何一项 |
| 2、3、9、10 | **有源文档支持** |

论断 6 值得细究。一个仓促的单个智能体常常忽略“增长最快的”与“增长最快的之一”之间的差别。一个上下文中别无他物的专职核查者，再经过质疑者挑战，就很难被蒙混过关。这种精确性来自工作流的结构，而不是来自更聪明的模型。

如果工作流在有依据的论断上比我们的答案更严格，不要惊讶。论断 3、9 和 10 正是质疑环节倾向于挑战的那类：调研确认了 12400 这个数字，但没有源文档确立“增长”此前的基线（论断 3）；新闻文章说公司“近期扩展”到三个品类，但没有确认这些发布发生在*Q2 之内*（论断 9）；虽然 5 月没有丢失数据，但质疑者可能反对把一次 6 小时中断称为“短暂的”干扰（论断 10）。无论你的运行是确认还是标记了它们，那都是对抗性核查在发挥作用。过度严格的标记只花费一分钟人工复核；不够严格的通过则会损失投资人的信任。

### 阅读 Claude 编写的脚本

最有启发性的产物是脚本本身，因为它展示了工作是如何组织的。`Workflow` 工具结果中包含持久化脚本的路径，`run_agent()` 捕获了它。阅读脚本时，注意强制核查者返回结构化判定的 `schema` 选项，以及各阶段之间朴素的 JavaScript 逻辑：只把已确认的判定路由给质疑者，词元成本为零。

In [7]:
show_script(factcheck_info)

```javascript
export const meta = {
  name: 'orbitcart-fact-check',
  description: 'Fact-check the OrbitCart investor update against source documents',
  phases: [
    { title: 'Extract' },
    { title: 'Verify' },
    { title: 'Skeptic' },
    { title: 'Report' },
  ],
}

const EXTRACT_SCHEMA = {
  type: 'object',
  properties: {
    claims: {
      type: 'array',
      items: {
        type: 'object',
        properties: {
          number: { type: 'integer' },
          text: { type: 'string' },
        },
        required: ['number', 'text'],
      },
    },
  },
  required: ['claims'],
}

const VERIFY_SCHEMA = {
  type: 'object',
  properties: {
    number: { type: 'integer' },
    text: { type: 'string' },
    verdict: { type: 'string', enum: ['confirmed', 'contradicted', 'unverifiable'] },
    source_file: { type: 'string', description: 'relative path of the source file relied on, or empty string' },
    quote: { type: 'string', description: 'exact quoted line(s) from the source relied on; empty string if unverifiable' },
    correct_figure: { type: 'string', description: 'the correct figure/fact per the source, only if contradicted; else empty string' },
    reasoning: { type: 'string' },
  },
  required: ['number', 'text', 'verdict', 'source_file', 'quote', 'correct_figure', 'reasoning'],
}

const SKEPTIC_SCHEMA = {
  type: 'object',
  properties: {
    number: { type: 'integer' },
    final_verdict: { type: 'string', enum: ['confirmed', 'contradicted'] },
    source_file: { type: 'string' },
    quote: { type: 'string' },
    correct_figure: { type: 'string', description: 'only if downgraded to contradicted; else empty string' },
    reasoning: { type: 'string' },
  },
  required: ['number', 'final_verdict', 'source_file', 'quote', 'correct_figure', 'reasoning'],
}

// 1. EXTRACT
phase('Extract')
const extraction = await agent(
  `Read the file investor_update.md in the current working directory. It is a draft investor update ` +
  `containing a numbered list of ten highlights (1-10). Extract each numbered highlight as exactly one ` +
  `claim, preserving the draft's own numbering 1-10. Treat each numbered item as exactly one claim even ` +
  `when it bundles two figures together (e.g. an item that states both a dollar amount and a percentage ` +
  `is still ONE claim, not two). Return the claim text verbatim as written in the draft (including bolded ` +
  `figures), for all ten items.`,
  { label: 'extract', schema: EXTRACT_SCHEMA }
)

const claims = extraction.claims.sort((a, b) => a.number - b.number)
log(`Extracted ${claims.length} claims from investor_update.md`)

// 2. VERIFY (parallel, one per claim) then 3. SKEPTIC (only for confirmed verdicts)
const results = await pipeline(
  claims,
  (claim) => agent(
    `You are fact-checking one highlight from OrbitCart's draft Q2 2026 investor update against source ` +
    `documents. The source documents live in the sources/ directory of the current working directory: ` +
    `sources/monthly_sales.csv, sources/customer_survey.txt, sources/press_coverage.md, and ` +
    `sources/uptime_report.txt. Read all four files.\n\n` +
    `Claim #${claim.number}: "${claim.text}"\n\n` +
    `Determine a verdict:\n` +
    `- "confirmed" if a source directly supports the claim as written. Quote the exact supporting line(s).\n` +
    `- "contradicted" if a source conflicts with the claim (a different number, a different attribution, ` +
    `a stat that doesn't match, etc). Quote the exact conflicting line(s) and state the correct figure/fact ` +
    `per the source.\n` +
    `- "unverifiable" if no source covers this claim at all.\n\n` +
    `Be precise and skeptical: pay close attention to subtle differences between what a source actually ` +
    `says and what the draft claims it says (e.g. a claim may bundle a true figure with a false one, cite ` +
    `the wrong quarter, misattribute a quote, round in a misleading way, or overstate what a source covers). ` +
    `If the claim bundles two figures and only one is supported, that counts as "contradicted", not ` +
    `"confirmed" — quote the source and explain exactly which part is wrong. Quote exact lines from the ` +
    `source file(s), do not paraphrase the quote field.`,
    { label: `verify-${claim.number}`, phase: 'Verify', schema: VERIFY_SCHEMA }
  ),
  (verifyResult, claim) => {
    if (!verifyResult || verifyResult.verdict !== 'confirmed') {
      return Promise.resolve(verifyResult ? { ...verifyResult, skeptic_reviewed: false } : null)
    }
    return agent(
      `You are a skeptical fact-checking reviewer. Another verifier examined claim #${claim.number} from ` +
      `OrbitCart's draft Q2 2026 investor update and marked it "confirmed", citing this evidence:\n\n` +
      `Claim: "${claim.text}"\n` +
      `Cited source file: ${verifyResult.source_file}\n` +
      `Cited quote: "${verifyResult.quote}"\n` +
      `Verifier reasoning: ${verifyResult.reasoning}\n\n` +
      `Re-read the cited source file (path relative to the current working directory, inside sources/) ` +
      `yourself and try hard to REFUTE this confirmation. Look for: the quote being taken out of context, ` +
      `the quote actually supporting a different number/fact than the claim states, the claim bundling a ` +
      `second figure that the quote does NOT cover, misattribution, wrong time period, or rounding that ` +
      `changes the meaning. If you find the citation does NOT actually fully support the claim, set ` +
      `final_verdict to "contradicted", quote the real conflicting/limiting line, and state the correct ` +
      `figure or fact. Only keep final_verdict "confirmed" if the citation genuinely and fully supports the ` +
      `claim as written after your adversarial check.`,
      { label: `skeptic-${claim.number}`, phase: 'Skeptic', schema: SKEPTIC_SCHEMA }
    ).then((skepticResult) => {
      if (!skepticResult) return { ...verifyResult, skeptic_reviewed: false }
      if (skepticResult.final_verdict === 'contradicted') {
        return {
          ...verifyResult,
          verdict: 'contradicted',
          source_file: skepticResult.source_file || verifyResult.source_file,
          quote: skepticResult.quote || verifyResult.quote,
          correct_figure: skepticResult.correct_figure || verifyResult.correct_figure,
          reasoning: `Skeptic overturned confirmation: ${skepticResult.reasoning}`,
          skeptic_reviewed: true,
        }
      }
      return { ...verifyResult, skeptic_reviewed: true, reasoning: `${verifyResult.reasoning} (skeptic upheld: ${skepticResult.reasoning})` }
    })
  }
)

const finalResults = results.filter(Boolean).sort((a, b) => a.number - b.number)
log(`Verification complete for ${finalResults.length} claims`)

// 4. REPORT
phase('Report')
const report = await agent(
  `Compile a markdown fact-check report for OrbitCart's draft Q2 2026 investor update ` +
  `(investor_update.md), based on this verification data (JSON, one entry per numbered claim, already ` +
  `including any skeptic-review overturns):\n\n${JSON.stringify(finalResults, null, 2)}\n\n` +
  `Produce a markdown report with:\n` +
  `1. A table with columns: Claim #, Claim, Verdict, Evidence (the source file and quote, or "no source ` +
  `covers this" for unverifiable claims, or the conflicting quote + correct figure for contradicted claims).\n` +
  `2. A "Summary" section listing exactly what must be fixed before the update can be sent — call out every ` +
  `"contradicted" claim with the correct figure/fact to use instead, and every "unverifiable" claim that ` +
  `needs a source or should be softened/removed. If everything is confirmed, say so plainly.\n\n` +
  `Return only the finished markdown report as your final answer.`,
  { label: 'report' }
)

return report

```

### 如何阅读工作流脚本

每个工作流脚本都使用相同的基本构件。完整的选项集在 [Agent SDK 参考](https://code.claude.com/docs/en/agent-sdk/typescript) 的 Workflow 工具条目中有文档说明，但把常见原语与上面所见对应起来，你就能掌握大部分内容：

| 原语 | 作用 |
|---|---|
| `export const meta = {name, description, phases}` | 声明工作流的名称和各阶段；进度界面按这些阶段把智能体分组 |
| `agent(prompt, options)` | 派生一个具有**干净上下文**的子智能体：它只能看到给它的提示词字符串，别无其他。选项可以设置 `label`、`phase`、用于结构化输出的 JSON `schema` 和 `model` |
| `parallel([...])` | 并发运行一批智能体，并在继续之前**等待全部完成**（屏障） |
| `pipeline(items, stage1, stage2, ...)` | 让每个条目**独立**通过一系列阶段，因此条目 A 可以处于阶段 2 而条目 B 仍在阶段 1 |
| `phase("...")` | 标出随后的智能体属于哪个阶段 |
| 调用之间的普通 JavaScript | 过滤、去重、合并和循环，全部精确、即时、不产生词元成本 |
| `return {...}` | 脚本返回什么，就回传到你的会话 |

这一设计带来三个特性：

1. **计划由代码强制执行。** 质疑环节在核查者之后运行，是因为脚本的控制流如此排序。一切都不依赖 Claude 记住某条指令。
2. **Claude 的上下文保持干净。** 每项判定和引用的源文档行都存在于脚本变量中。只有最终汇编的报告到达了启动工作流的会话。
3. **脚本属于你。** 它是磁盘上的一个文件。你可以阅读它（我们刚刚做了）、编辑它、把它保存到项目的 `.claude/workflows/` 目录，并按名称重跑。编排成为团队可复用的资产。

## 何时选择工作流

动态工作流位于难度曲线的顶端；在其之下，单个智能体和子智能体仍是合适的工具。一个合理的决策指南：

| 何时选用…… | 适用场景…… |
|---|---|
| **单个智能体** | 任务能装进一个上下文窗口，且不需要独立核查。大多数任务属于这一类。|
| **子智能体** | 你需要几个专职工作者（一个研究员、一个评审员），并且主控智能体应根据结果边做边调整计划 |
| **动态工作流** | 任务超出单个上下文窗口的容量（条目多、来源多），需要不可省略的核查，或者这是一个你会重复执行的流程 |

三条值得从本笔记本保留的成本控制习惯：

1. **在提示词中明确描述结构。** 你描述的框架就是你得到的框架，包括运行多少核查、哪些阶段可以用更便宜的模型。
2. **关注每次运行的成本。** 上面的运行打印了一个成本数字。什么都核查的工作流比什么都不核查的工作流更贵，而这个调节旋钮由提示词设定。
3. **保持数据规模真实可信。** 我们用了 10 条论断和四个来源，使本笔记本能在几分钟内跑完。同样的结构可以扩展到数百个条目。脚本不在意规模，你的词元预算在意。

## 清理

删除本笔记本创建的工作区。如果你想先查看这些文件和事实核查证据，请跳过此单元格，并在再次运行事实核查之前重新运行创建工作区的单元格。

In [8]:
shutil.rmtree(WORKSPACE, ignore_errors=True)
print("Workspace removed.")

Workspace removed.


## 你学到的内容

对照引言中的目标回顾：

- **何时使用工作流：** 当任务超出单个上下文窗口、当核查必须是结构性的而非可选的、或者当这套编排值得保留时。在这条线之下，单个智能体和子智能体仍是合适的工具。
- **如何从 Agent SDK 触发工作流：** 在 `allowed_tools` 中允许 `Workflow` 工具，在提示词中要求使用工作流，并通过 `query()` 流式处理整个运行（启动、进度事件、完成）。
- **如何阅读 Claude 编写的脚本：** `meta` 声明各阶段，`agent()` 派生干净上下文的执行者，`parallel()` 是一道屏障，`pipeline()` 让条目流式通过各阶段，其间的逻辑由朴素的 JavaScript 免费处理。事实核查用的是**扇出 + 对抗性核查**；同样的原语可以组合成流水线、锦标赛和分诊队列。
- **如何让工作流可复用：** 脚本就是一个文件。把它保存到项目的 `.claude/workflows/` 目录并提交，之后任何在该项目中工作的人（或智能体）都能按名称重跑同一套编排。

## 下一步

- **把这一模式用到你自己的数据上。** 事实核查工作流适用于任何“论断对来源”的问题：报告、营销文案、文档。同样的结构（抽取、并行核查、质疑、汇编）可迁移到内容迁移、分诊队列以及“按判断力给大量事物排序”的问题。
- **在 Claude Code CLI 中尝试工作流**，在那里你可以用 `/workflows` 交互式观看运行，并用一次按键保存它们。参见[动态工作流文档](https://code.claude.com/docs/en/workflows)。
- **与本系列中的其他编排方式对比：** [笔记本 01](01_The_chief_of_staff_agent.ipynb) 深入讲解子智能体，而 [Building Effective Agents 模式集](../patterns/agents/README.md) 展示了直接基于 Messages API 实现的相同编排思想。
- **阅读 Anthropic 关于多智能体系统的工程博客** [How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)，了解多智能体配置何时胜过单个智能体及其代价背后的研究。